In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

: 

In [ ]:
# 01 - CONFIGURATION & MODEL SELECTOR

# Change this to: "resnet50", "efficientnet_b0", or "vit"
MODEL_CHOICE = "vit"
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
LOCAL_FOLDER = r"C:\Users\HaadiyaH\Desktop\Data\EndoThesisData"
CSV_PATH = r"C:\Users\HaadiyaH\Desktop\Data\mri_triple_reader_master.csv"

In [ ]:
# 02 - DATA PREP & PATIENT-WISE SPLIT

df = pd.read_csv(CSV_PATH)
df['file_path'] = df['file_path'].apply(lambda x: os.path.join(LOCAL_FOLDER, os.path.basename(x)))

# Ensure patient-wise separation (Medical Gold Standard)
unique_patients = df['patient_id'].unique()
train_ids, test_ids = train_test_split(unique_patients, test_size=0.20, random_state=42)
train_df = df[df['patient_id'].isin(train_ids)].reset_index(drop=True)
test_df = df[df['patient_id'].isin(test_ids)].reset_index(drop=True)

In [ ]:
# 03 - DATASET & AUGMENTATION
# Note: Heavy augmentation to make the most of your 2,100 slices
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

class EndoDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, torch.tensor(row['label'], dtype=torch.long)

# Handling Class Imbalance with Weighted Sampler
counts = train_df['label'].value_counts().sort_index().values
weights = 1.0 / torch.tensor(counts, dtype=torch.float)
sample_weights = [weights[l] for l in train_df['label']]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(EndoDataset(train_df, data_transforms['train']), batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(EndoDataset(test_df, data_transforms['test']), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 04 - BENCHMARK MODEL FACTORY

def build_benchmarking_model(name):
    if name == "resnet50":
        model = models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 2))
    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(weights='IMAGENET1K_V1')
        model.classifier[1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.classifier[1].in_features, 2))
    elif name == "vit":
        model = models.vit_b_16(weights='IMAGENET1K_V1')
        model.heads.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.heads.head.in_features, 2))
    return model.to(DEVICE)

model = build_benchmarking_model(MODEL_CHOICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# 05 - TRAINING LOOP WITH BEST MODEL SAVING

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), f'best_{MODEL_CHOICE}.pth')


In [ ]:
# 06 - FINAL EVALUATION (THESIS METRICS)

def evaluate_performance(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_labels.extend(lbls.cpu().numpy())
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    print(f"\nReport for {MODEL_CHOICE}:")
    print(classification_report(all_labels, all_preds, target_names=['Healthy', 'Endo']))
    return all_labels, all_preds

evaluate_performance(model, test_loader)